# Making a mapping movie

## Introduction

As part of working my way through the Udemy course [Udemy 30-Day Map Challenge 2025: Director's Cut](https://www.udemy.com/course/30-day-map-challenge-2025-directors-cut/learn/lecture/55045243#overview), there was a small segment on making animated maps.

I ran across a few gotchas in the process, and learned about a new library __PyAC__.

The essential process is to use __matplotlib__ to produce a number of JPEG images of a map (each image to be a frame in an animation).  We then stitch these into an MP4 file.  I will go through the process below, and then show some of the pitfall I fells into

## Implementation

## Package imports

- os used for file access
- imageio.v3 used for image management

In [5]:
import os
import imageio.v3 as iio

from IPython.display import IFrame


In [6]:
%load_ext watermark

------------------------------------------

## Images

Assume we have alreay created a folder ___frames_neon___ with the images to made into a map.

Note that the images are stored a JPG files:  when I initially used PNG files, I came unstuck.  The PNG files were 4 planes deep (I assume RGB+alpha), and the movie making software assumed 3 planes (see below).

List the image file names

In [7]:
folder = "frames_neon"

files = sorted([f for f in os.listdir(folder) if f.endswith(".jpg")])
files[0:5]

['frame_0000.jpg',
 'frame_0001.jpg',
 'frame_0002.jpg',
 'frame_0003.jpg',
 'frame_0004.jpg']

Get a list of images in the correct order

In [8]:
images = [iio.imread(os.path.join(folder, f)) for f in files]

### Making MP4

We use a plugin __pyav__.  In their own words:

    PyAV is a Pythonic binding for the FFmpeg libraries. We aim to provide all of the power and control of the underlying library, but manage the gritty details as much as possible.

We write the movie to a file "neon.mp4".  Note the code to make the dimension of the images (height, width) even numbers, and the line of code setting a __time_base__ variable

In [9]:

video_writer = iio.imopen("neon.mp4", "w", plugin="pyav")
video_writer.init_video_stream(codec="libx264", fps=20)

# Refer to
# https://github.com/imageio/imageio/issues/1139
video_writer._video_stream.codec_context.time_base = video_writer._video_stream.time_base

l_x, l_y, _ = images[0].shape
l_x = (l_x//2)*2
l_y = (l_y//2)*2

for image in images:
 video_writer.write_frame(image[0:l_x,0:l_y,])
#end for
video_writer.close()

Display MP4 player snapshot in notebook

In [10]:
IFrame("images/map_movie.png", width=800, height=400)

---------------------------------

## Pitfalls

### Odd image sizes

This first example shows the result of not cropping images height and width to get an even number.  The diagnostic must be close to taking the record as the most unhelpful message ever

In [11]:
video_writer = iio.imopen("bad.mp4", "w", plugin="pyav")
video_writer.init_video_stream(codec="libx264", fps=20)

# Refer to
# https://github.com/imageio/imageio/issues/1139
video_writer._video_stream.codec_context.time_base = video_writer._video_stream.time_base

l_x, l_y, _ = images[0].shape

for image in images:
 video_writer.write_frame(image[0:l_x,0:l_y,])
#end for
video_writer.close()

ExternalError: [Errno 542398533] Generic error in an external library: 'avcodec_open2("libx264", {})'

--------------------

### Not setting time_base

Setting the __time_base__ (it appears) is a workaround for a bug that was introduced in __pyav__ 15.0, and seems to be still there at __pyav__ 17.0.0.

Again, the diagnostic is also not very helpful

In [12]:
video_writer = iio.imopen("bad.mp4", "w", plugin="pyav")
video_writer.init_video_stream(codec="libx264", fps=20)

# Refer to
# https://github.com/imageio/imageio/issues/1139
# video_writer._video_stream.codec_context.time_base = video_writer._video_stream.time_base

l_x, l_y, _ = images[0].shape
l_x = (l_x//2)*2
l_y = (l_y//2)*2

for image in images:
 video_writer.write_frame(image[0:l_x,0:l_y,])
#end for
video_writer.close()

AttributeError: 'NoneType' object has no attribute 'numerator'

-------------------------------

### using PNG images

I don't know why __matplotlib uses__ 4 planes for PNG images, and 3 planes for JPG



In [13]:
folder = "frames"

files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])
files[0:5]

['frame_000.png',
 'frame_001.png',
 'frame_002.png',
 'frame_003.png',
 'frame_004.png']

In [14]:
images = [iio.imread(os.path.join("frames", f)) for f in files]

print(images[0].shape)

(2000, 2000, 4)


In [15]:

video_writer = iio.imopen("bad.mp4", "w", plugin="pyav")
video_writer.init_video_stream(codec="libx264", fps=20)

# Refer to
# https://github.com/imageio/imageio/issues/1139
video_writer._video_stream.codec_context.time_base = video_writer._video_stream.time_base

l_x, l_y, _ = images[0].shape
l_x = (l_x//2)*2
l_y = (l_y//2)*2

for image in images:
 video_writer.write_frame(image[0:l_x,0:l_y,])
#end for
video_writer.close()

ValueError: could not broadcast input array from shape (2000,2000,4) into shape (2000,2000,3)

-------------------

## Alternatives

An alternative to using `imageio.v3` would be to use the OpenCV library (`import cv2`) , with code that looks like:

    # Define the codec and create VideoWriter object
    # 'mp4v' is a standard codec for mp4 format
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_file, fourcc, fps, size)

    ...

    # Write each frame to the video
    for i, file_path in enumerate(image_files):
        img = cv2.imread(file_path)
        out.write(img)

----------------------

## Reproducability

In [16]:
%watermark

Last updated: 2026-04-14T13:39:57.737213+10:00

Python implementation: CPython
Python version       : 3.11.15
IPython version      : 9.10.1

Compiler    : MSC v.1944 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 170 Stepping 4, GenuineIntel
CPU cores   : 22
Architecture: 64bit



In [17]:
%watermark -h -iv -co

conda environment: 30DayMapChallenge2025

Hostname: INSPIRON16

IPython: 9.10.1
imageio: 2.37.0



In [18]:
import contextlib

import ipynbname

with contextlib.suppress(FileNotFoundError):
    print(f"Notebook file name: {ipynbname.name()}")
# end with

Notebook file name: making_movie
